<a href="https://colab.research.google.com/github/rajidush/RP-J26-DS-316/blob/c3%2Fmodel-tuning/component3_socratic_educator/notebooks/train_inquire_lora_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/rajidush/RP-J26-DS-316/blob/c3%2Fmodel-tuning/component3_socratic_educator/notebooks/train_inquire_lora_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train the INQUIRE LoRA adapter (Component 3)

Fine-tunes `google/gemma-3-1b-it` with the **fixed** LoRA config in `inquire_training.py`
(r=16, alpha=32, attention + MLP projections, LR 1e-4, 3 epochs, completion-only loss),
evaluating and saving every epoch and keeping the checkpoint with the lowest validation loss.
There is no hyperparameter search.

**Before you start**
- Runtime → Change runtime type → **T4 GPU**.
- Colab secret `HF_TOKEN` (🔑 in the sidebar) for a Hugging Face account that has accepted the Gemma licence.
- From the repo: `component3_socratic_educator/inquire_training.py` and the four files in
  the `build` folder named by `DATASET_FOLDER` below, e.g. `component3_socratic_educator/evidence/inquire_dataset_2026-10-10/` (`manifest.json`, `train.jsonl`, `validation.jsonl`, `test.jsonl`).

**Outputs** go to Google Drive only (`MyDrive/c3_inquire_training/<run id>/`): checkpoints, the adapter and `run_record.json`.
Only `run_record.json` comes back to the repo, as evidence. The adapter never does.

In [1]:
!pip -q install "transformers==4.56.2" "trl==0.23.1" "peft==0.17.1" "accelerate==1.10.1" "datasets==4.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This beha

In [2]:
import time
from google.colab import drive
drive.mount("/content/drive")

RUN_ID = time.strftime("%Y-%m-%d_%H%M", time.gmtime())
RUN_DIR = f"/content/drive/MyDrive/c3_inquire_training/{RUN_ID}"
ADAPTER_DIR = f"{RUN_DIR}/adapter"
DATA_DIR = "/content/inquire_dataset"
DATASET_FOLDER = "inquire_dataset_2026-10-10"   # the repo `build` folder you upload from (recorded in the run record)
print("run directory:", RUN_DIR)

Mounted at /content/drive
run directory: /content/drive/MyDrive/c3_inquire_training/2026-10-10_1249


Upload `inquire_training.py`, `manifest.json`, `train.jsonl`, `validation.jsonl` and `test.jsonl` (select all five at once).

In [3]:
import os
import re
from google.colab import files

EXPECTED = ("inquire_training.py", "manifest.json", "train.jsonl", "validation.jsonl", "test.jsonl")
uploaded = files.upload()
os.makedirs(DATA_DIR, exist_ok=True)
for name in uploaded:
    base = re.sub(r" \(\d+\)(?=\.)", "", name)  # Colab renames re-uploads to 'file (1).jsonl'
    if base not in EXPECTED:
        print("ignored:", name)
        continue
    os.replace(name, base if base == "inquire_training.py" else f"{DATA_DIR}/{base}")
missing = [n for n in EXPECTED if not os.path.exists(n if n.endswith(".py") else f"{DATA_DIR}/{n}")]
assert not missing, f"still missing: {missing}"
print("uploaded:", ", ".join(EXPECTED))

Saving inquire_training.py to inquire_training.py
Saving manifest.json to manifest.json
Saving test.jsonl to test.jsonl
Saving train.jsonl to train.jsonl
Saving validation.jsonl to validation.jsonl
uploaded: inquire_training.py, manifest.json, train.jsonl, validation.jsonl, test.jsonl


Stop here unless the dataset is exactly the `build` output its manifest describes.

In [4]:
import inquire_training as tr

manifest = tr.verify_dataset(DATA_DIR)   # raises DatasetError on a missing or mismatched file
MANIFEST_SHA256 = tr.manifest_sha256(DATA_DIR)
for split in ("train", "validation", "test"):
    c = manifest["counts"][split]
    print(f"{split:<10} {c['rows']:>3} rows, {c['scenarios']:>2} scenarios")
print("manifest sha256:", MANIFEST_SHA256)

train      104 rows, 41 scenarios
validation  25 rows,  5 scenarios
test        50 rows, 10 scenarios
manifest sha256: 444a5aae4fc83ec74cc0d1dc4e757a976b7725d00c1eb5cb004fcde701f46eba


In [5]:
import torch
from google.colab import userdata
from huggingface_hub import login

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
GPU = torch.cuda.get_device_name(0)
login(token=userdata.get("HF_TOKEN"))
print("GPU:", GPU, "| precision:", tr.PRECISION)

GPU: Tesla T4 | precision: fp32


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

set_seed(tr.SEED)
tokenizer = AutoTokenizer.from_pretrained(tr.BASE_MODEL)
# fp32 (see tr.PRECISION_REASON); eager attention is the recommended setting for training Gemma 3.
model = AutoModelForCausalLM.from_pretrained(tr.BASE_MODEL, dtype=torch.float32, attn_implementation="eager")
MODEL_REVISION = getattr(model.config, "_commit_hash", None)
print("model revision:", MODEL_REVISION)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

model revision: dcc83ea841ab6100d6b47a070329e1ba4cf78752


In [8]:
from datasets import load_dataset

data = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl",
                                        "validation": f"{DATA_DIR}/validation.jsonl"})
data = data.select_columns(["prompt", "completion"])   # metadata stays in the files, not in training
lengths = [len(tokenizer.apply_chat_template(r["prompt"] + r["completion"], tokenize=True))
           for split in data.values() for r in split]
MAX_TOKENS = max(lengths)
assert MAX_TOKENS <= tr.TRAINER_CONFIG["max_length"], f"{MAX_TOKENS} tokens would be truncated"
print(data)
print("longest prompt + completion:", MAX_TOKENS, "tokens")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 104
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 25
    })
})
longest prompt + completion: 259 tokens


In [9]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

args = SFTConfig(output_dir=f"{RUN_DIR}/checkpoints", **tr.TRAINER_CONFIG)
trainer = SFTTrainer(model=model, args=args, train_dataset=data["train"], eval_dataset=data["validation"],
                     processing_class=tokenizer, peft_config=LoraConfig(**tr.LORA_CONFIG))
assert "completion_mask" in trainer.train_dataset.column_names, "loss would not be completion-only"
TRAINABLE = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
trainer.model.print_trainable_parameters()

Tokenizing train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


In [10]:
import math

STARTED_AT = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
start = time.time()
trainer.train()
RUNTIME = round(time.time() - start, 1)

LOSSES = tr.epoch_losses(trainer.state.log_history)
for row in LOSSES:
    print(row)
bad = [e for e in LOSSES if not (isinstance(e["eval_loss"], float) and math.isfinite(e["eval_loss"]))]
assert not bad, f"missing or non-finite validation loss: {bad}"
print("kept:", trainer.state.best_model_checkpoint, "eval_loss", trainer.state.best_metric, "| runtime", RUNTIME, "s")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,2.050500,1.060433,2.010008,24900.000000,0.695601
2,0.992900,0.877255,1.790274,49800.000000,0.762511
3,0.755100,0.850411,1.661432,74700.000000,0.765955


{'epoch': 1, 'step': 26, 'train_loss': 2.0505, 'eval_loss': 1.0604325532913208}
{'epoch': 2, 'step': 52, 'train_loss': 0.9929, 'eval_loss': 0.8772547245025635}
{'epoch': 3, 'step': 78, 'train_loss': 0.7551, 'eval_loss': 0.8504109382629395}
kept: /content/drive/MyDrive/c3_inquire_training/2026-10-10_1249/checkpoints/checkpoint-78 eval_loss 0.8504109382629395 | runtime 191.4 s


`load_best_model_at_end` has already put the lowest-validation-loss checkpoint back into `trainer.model`; save that adapter.

In [11]:
trainer.save_model(ADAPTER_DIR)
print("adapter saved to Drive:", ADAPTER_DIR)

adapter saved to Drive: /content/drive/MyDrive/c3_inquire_training/2026-10-10_1249/adapter


In [12]:
import importlib.metadata as md
import json
import platform

def version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None

versions = {p: version(p) for p in ("torch", "transformers", "trl", "peft", "accelerate", "datasets")}
versions["python"] = platform.python_version()

record = tr.run_record(
    manifest_sha256=MANIFEST_SHA256, dataset_folder=DATASET_FOLDER, versions=versions, gpu=GPU,
    runtime_seconds=RUNTIME, log_history=trainer.state.log_history,
    best_checkpoint=trainer.state.best_model_checkpoint, best_metric=trainer.state.best_metric,
    adapter_location=ADAPTER_DIR,
    extra={"run_id": RUN_ID, "started_at": STARTED_AT, "model_revision": MODEL_REVISION, "cuda": torch.version.cuda,
           "train_rows": len(data["train"]), "validation_rows": len(data["validation"]),
           "max_prompt_completion_tokens": MAX_TOKENS, "trainable_parameters": TRAINABLE,
           "pinned_packages": list(tr.PINNED_PACKAGES), "trainer_args_full": json.loads(args.to_json_string())})
tr.write_run_record(f"{RUN_DIR}/run_record.json", record)
files.download(f"{RUN_DIR}/run_record.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**After the run:** save the downloaded `run_record.json` as
`component3_socratic_educator/evidence/inquire_training_<date>/run_record.json` in the repo.
The adapter stays on Drive (`ADAPTER_DIR`); ticket 06 loads it from there for the comparison.